# INFO 2950 Phase III — Data cleaning
Jiaxin Liu (jl4885), Xi Lin (xl838), Amanda Lu (xl744)

This notebook starts from the saved raw responses and the supplied matching/official-check reference tables. It is based on our submitted Phase II code. It exports clean CSVs used by project_phase_III.ipynb; the analysis notebook does not repeat raw-data cleaning.

## Missing values and duplicates: rules before analysis
We check **every column** at each table stage, counting both NA and blank strings in `missingness_by_stage.csv`. Zero gender codes are not NA: they mean unspecified. Invalid gender codes stop the run. We preserve unknown gender without guessing it or classifying it as male.

Required film titles, years and movie IDs must be present and unique. An unresolved movie match or duplicate film key stops for review; we never choose an arbitrary film. Missing actor IDs are logged and excluded because we cannot deduplicate a person without an ID. Missing billing order stops for review. Within each film, repeated person IDs are sorted and only the earliest credit is kept; cross-film repeats are valid. Exclusions and duplicate records are saved.

Optional actor names/roles and descriptive metadata are retained even if missing; they do not define the modeling sample. Candidate release-date gaps remain missing and are not filled with zero. Missing gender becomes code 0. A zero denominator makes a ratio missing, not zero. In particular, women_to_men_ratio is missing for a selected cast with no actors recorded as male; we retain the film because this ratio is not used in any model.

For each model, required missing inputs/outcomes would exclude that film **only from that model**. This is done here and recorded in model_eligibility.csv, with separate rq1/rq2/rq3_ready.csv files. We do not drop a film because an unrelated field is missing. The 80% coverage restriction is a planned sensitivity check, not a replacement for the full sample.

In [1]:
# Read the supplied JSON and parse its HTML using the discussion approach.
import json
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)
REF = Path("data/reference")
# Read the saved response, then extract its HTML field.
raw = json.loads((RAW / "best_picture.json").read_text())
page = BeautifulSoup(raw["html"], "html.parser")
START_YEAR, END_YEAR = 2000, 2025
print("Wikipedia revision:", raw["latest"])

# Record NA and blank counts for every column, without modifying optional fields.
audit_rows = []
def audit_frame(stage, frame):
    for col in frame.columns:
        blank = frame[col].map(lambda v: isinstance(v, str) and v.strip() == "")
        audit_rows.append({"stage": stage, "column": col, "rows": len(frame),
                           "na_count": int(frame[col].isna().sum()),
                           "blank_count": int(blank.sum())})
def require_fields(frame, fields):
    for col in fields:
        assert frame[col].notna().all(), f"Missing required field: {col}"
        assert not frame[col].map(lambda v: isinstance(v, str) and v.strip() == "").any(), col

Wikipedia revision: {'id': 1375974144, 'timestamp': '2026-09-21T05:43:50Z'}


**2. Get the nominees and winners.** First, we use a `for` loop, `find()`, `find_all()` and an `if` statement to find the nomination tables and read film titles and links. `recursive=False` limits the cell search to the current row. Next, we set a nested `for` loop and again use `find()` and `find_all()` to find the ceremony year, film title and winning status. 

Several things to note: For year, on the Wikipedia page, a year heading covers several film rows, so we use that ceremony number until the next heading appears. For these years, the ceremony year is the ceremony number (`ceremony`) plus 1928. We keep the film eligibility-year label (`year_label`) separately. For the film title, all titles are in italicized link form, so we use tags `i` and `a`. For winning status, in the wiki page, the gold row shading identifies winners, so we used `in` to check and return a boolean value, and the bold title formatting provides a second extraction check. 

Once we get all the information needed, we append the information film by film to `records`, and finally convert it to a dataframe `nominees` with rows sorted first by ceremony years and then by title.

This gives 196 nominees across 26 ceremonies, saved in `01_wikipedia_nominees.csv`. We check for duplicate films, missing years and exactly one winner per year. We also checked all 23 nominees from the 2000, 2021 and 2025 ceremonies against Academy records. Their titles and winner status agreed. Sources are in `data/reference/academy_spot_checks.csv`; this check does not cover every year or the TMDB cast records.

In [2]:
# Rebuild one row per nominated film from the original Wikipedia HTML.
records = []
for table in page.find_all("table", {"class": "wikitable"}):
    headings = [cell.get_text(" ", strip=True)
                for cell in table.find("tr").find_all("th")]
    if "Year of film release" not in headings or "Film" not in headings:
        continue
    # A merged year heading applies to multiple film rows.
    ceremony = None
    year_label = None
    for row in table.find_all("tr"):
        year_cell = row.find("th", {"rowspan": True})
        if year_cell is not None:
            for link in year_cell.find_all("a"):
                href = link.get("href", "")
                if "Academy_Awards" in href:
                    digits = "".join(c for c in link.get_text() if c.isdigit())
                    ceremony = int(digits)
                if "_in_film" in href:
                    year_label = link.get_text()
        if ceremony is None or not START_YEAR <= ceremony + 1928 <= END_YEAR:
            continue
        fields = row.find_all("td", recursive=False)
        if not fields or fields[0].find("i") is None:
            continue
        film = fields[0].find("i").find("a")
        if film is None:
            continue
        # Winner shading and bold text should identify the same row.
        winner = "faeb86" in row.get("style", "").lower()
        assert winner == (fields[0].find("b") is not None)
        records.append({
            "ceremony_number": ceremony,
            "eligibility_year_label": year_label,
            "film_title": film.get_text().strip(),
            "wiki_title": film.get("title"),
            "wiki_url": "https://en.wikipedia.org/wiki/" + film["href"].replace("./", "", 1),
            "winner": winner,
            "ceremony_year": ceremony + 1928
        })
nominees = pd.DataFrame(records).sort_values(["ceremony_year", "film_title"])
assert not nominees.duplicated(["ceremony_year", "wiki_url"]).any()
assert (nominees.groupby("ceremony_year")["winner"].sum() == 1).all()
assert nominees["ceremony_year"].nunique() == END_YEAR - START_YEAR + 1
nominees.to_csv(OUT / "01_wikipedia_nominees.csv", index=False)
official = pd.read_csv("data/reference/academy_spot_checks.csv")
checked_years = official["ceremony_year"].unique()
sample = nominees[nominees["ceremony_year"].isin(checked_years)]
assert len(sample) == len(official)
checked = sample.merge(official, on=["ceremony_year", "film_title"], how="left")
assert checked["official_winner"].notna().all()
assert (checked["winner"] == checked["official_winner"]).all()
print("Academy spot-check:", len(checked), "film records across", len(checked_years), "ceremonies agree")
print("Nominations:", len(nominees), "Ceremonies:", nominees["ceremony_year"].nunique())
display(nominees[["ceremony_year", "film_title", "winner"]].head())
audit_frame("nominees", nominees)
require_fields(nominees, ["film_title", "ceremony_year", "winner", "wiki_url"])

Academy spot-check: 23 film records across 3 ceremonies agree
Nominations: 196 Ceremonies: 26


,ceremony_year,film_title,winner
0,2000,American Beauty,True
1,2000,The Cider House Rules,False
2,2000,The Green Mile,False
3,2000,The Insider,False
4,2000,The Sixth Sense,False


**3. Read TMDB search results and popularity.** The saved file index tells us which search response belongs to each film. We first check that its title and year order matches our Wikipedia table. We then use `iterrows()` and a loop to put all search results into a table with 1,982 rows. We keep all possible matches because the same title may return a remake or a different film. A missing release year is saved as `np.nan`, not zero.

For each result, we keep its movie ID, titles, release date, **movie** popularity and collection time. Dictionary `.get()` lets us give a default value when a field is missing. We save the candidate films into the dataframe `candidates` that will be used for the following cleaning steps and save this table as `02_tmdb_candidates.csv` as well.

In [3]:
# Expand the saved search responses, including movie popularity and its timestamp.
collection_index = pd.read_csv(REF / "tmdb_movie_matches.csv")
assert collection_index["film_title"].tolist() == nominees["film_title"].tolist()
assert collection_index["ceremony_year"].tolist() == nominees["ceremony_year"].tolist()
manifest = json.loads((RAW / "tmdb/collection_manifest.json").read_text())
collected_at = {}
for entry in manifest:
    collected_at[Path(entry["file"]).name] = entry["collected_at_utc"]

# Preserve every search candidate; choose the correct movie in Step 4.
candidate_records = []
for _, row in collection_index.iterrows():
    response = json.loads((RAW / "tmdb" / row["search_file"]).read_text())
    for movie in response["results"]:
        date = movie.get("release_date", "")
        candidate_records.append({
            "ceremony_year": row["ceremony_year"], "film_title": row["film_title"],
            "search_file": row["search_file"], "tmdb_id": movie["id"],
            "tmdb_title": movie.get("title", ""),
            "original_title": movie.get("original_title", ""),
            "release_date": date, "release_year": int(date[:4]) if date else np.nan,
            "movie_popularity": movie.get("popularity", np.nan),
            "popularity_collected_at_utc": collected_at[row["search_file"]]
        })
candidates = pd.DataFrame(candidate_records)
candidates.to_csv(OUT / "02_tmdb_candidates.csv", index=False)
print("Candidate records:", len(candidates))
audit_frame("search_candidates", candidates)

Candidate records: 1982


**4. Find the correct TMDB movie.** We need to make titles easier to compare by removing differences in capital letters, accents and punctuation. So, we defined a `normalize_title` function that uses `unicodedata.normalize()` to help remove accents. `.map()` applies this function to each title. We keep the original titles too.

We then use a loop and `iterrows()` to go through the rows in `nominees`. If a film has a recorded review decision, we use that movie ID. Otherwise, we first look for one matching title released one or two years before the ceremony. If this does not give one result, we require a single title match. The review decisions cover exceptions, including the 2021 eligibility period. We check that only one result remains before using `.iloc[0]` to select it. We get 169 automatic matches and 27 reviewed matches, saved in the dataframe `matched` and `03_resolved_movie_matches.csv`. Each selected movie keeps its own popularity score.

In [4]:
# Apply the same title/year rules and supplied review decisions as the original package.
def normalize_title(title):
    text = unicodedata.normalize("NFKD", str(title)).encode("ascii", "ignore").decode()
    return "".join(c for c in text.lower() if c.isalnum())

for column in ["film_title", "tmdb_title", "original_title"]:
    candidates[column + "_key"] = candidates[column].map(normalize_title)
review = pd.read_csv(REF / "movie_match_decisions.csv")
assert not review.duplicated(["ceremony_year", "film_title"]).any()
matches = []
for _, row in nominees.iterrows():
    pool = candidates[(candidates["ceremony_year"] == row["ceremony_year"]) &
                      (candidates["film_title"] == row["film_title"])]
    decision = review[(review["ceremony_year"] == row["ceremony_year"]) &
                      (review["film_title"] == row["film_title"])]
    # Use a documented review decision when one exists.
    if len(decision) == 1:
        selected = pool[pool["tmdb_id"] == int(decision.iloc[0]["tmdb_id"])]
        method = "reviewed_title_date_synopsis"
    else:
        exact = pool[(pool["film_title_key"] == pool["tmdb_title_key"]) |
                     (pool["film_title_key"] == pool["original_title_key"])]
        plausible = exact[exact["release_year"].isin(
            [row["ceremony_year"] - 1, row["ceremony_year"] - 2])]
        selected = plausible if len(plausible) == 1 else exact
        method = "unique_title_year" if len(plausible) == 1 else "unique_title"
    # Never take the first result before checking that the match is unique.
    assert len(selected) == 1, row["film_title"]
    movie = selected.iloc[0]
    matches.append({
        "ceremony_year": row["ceremony_year"], "film_title": row["film_title"],
        "tmdb_id": int(movie["tmdb_id"]), "tmdb_title": movie["tmdb_title"],
        "tmdb_release_date": movie["release_date"], "match_method": method,
        "search_file": movie["search_file"],
        "movie_popularity": movie["movie_popularity"],
        "popularity_collected_at_utc": movie["popularity_collected_at_utc"]
    })
matched = pd.DataFrame(matches)
assert len(matched) == len(nominees)
assert not matched["tmdb_id"].duplicated().any()
matched.to_csv(OUT / "03_resolved_movie_matches.csv", index=False)
print("Resolved movies:", len(matched), "Documented review decisions:", len(review))
audit_frame("resolved_movies", matched)
require_fields(matched, ["tmdb_id", "film_title", "ceremony_year"])

Resolved movies: 196 Documented review decisions: 27


**5. Clean the cast and choose up to ten actors.** We read each movie's `cast` list. We keep actor IDs, names, roles, cast order and gender codes. Actors without IDs would be removed and recorded, but none are missing here. We sort by movie ID and cast order, using the original list position when orders are tied. If an order is missing, the code stops for us to check it.

One actor may have more than one role in a film. We remove repeated movie-ID and person-ID pairs so that each actor counts once per film. The same actor can still appear in another film. This removes 3 duplicate credits from 13,208 records. We then use `.head(10)` for each film and `pd.concat()` to put the groups together, and we save them into the dataframe `top10` with 1,955 actor-film rows. Two films have fewer than ten actors. We keep voice, self and uncredited roles if they appear within the selected positions in TMDB's cast order.

Gender codes 1, 2 and 3 mean female, male and non-binary. Code 0 or a missing value means unspecified. We keep these groups separate and do not guess anyone's gender. We save the cast tables at each stage and keep separate files for duplicate or missing-ID records.

In [5]:
# Flatten cast records, retain valid person IDs, and remove within-film duplicates.
cast_records = []
for _, movie in matched.iterrows():
    response = json.loads((RAW / "tmdb" / f"credits_{movie['tmdb_id']}.json").read_text())
    assert response["id"] == movie["tmdb_id"]
    for position, actor in enumerate(response["cast"]):
        gender = actor.get("gender")
        original_gender = gender
        if gender is None:
            gender = 0
        assert gender in [0, 1, 2, 3]
        cast_records.append({
            "tmdb_id": movie["tmdb_id"], "person_id": actor.get("id"),
            "actor_name": actor.get("name"), "character": actor.get("character"),
            "billing_order": actor.get("order"), "source_position": position,
            "credit_id": actor.get("credit_id"), "gender_code": gender,
            "gender_code_original": original_gender
        })
cast_raw = pd.DataFrame(cast_records)
cast_raw.to_csv(OUT / "04_cast_flattened.csv", index=False)
missing_ids = cast_raw[cast_raw["person_id"].isna()]
cast = cast_raw[cast_raw["person_id"].notna()].copy()
assert cast["billing_order"].notna().all()
# Order records before removing repeated people within the same film.
cast = cast.sort_values(["tmdb_id", "billing_order", "source_position"])
duplicates = cast[cast.duplicated(["tmdb_id", "person_id"], keep="first")]
cast = cast.drop_duplicates(["tmdb_id", "person_id"], keep="first")
cast["gender_category"] = cast["gender_code"].map(
    {0: "unspecified", 1: "female", 2: "male", 3: "nonbinary"})

cast.to_csv(OUT / "05_cast_cleaned.csv", index=False)

# Take ten different people only after within-film deduplication.
selected_groups = []
for _, movie in matched.iterrows():
    selected_groups.append(cast[cast["tmdb_id"] == movie["tmdb_id"]].head(10))
top10 = pd.concat(selected_groups, ignore_index=True)
top10.to_csv(OUT / "06_top10_cast.csv", index=False)
duplicates.to_csv(OUT / "audit_duplicate_person_credits.csv", index=False)
missing_ids.to_csv(OUT / "audit_missing_person_id.csv", index=False)
print("Cast records:", len(cast_raw), "Duplicates removed:", len(duplicates))
print("Missing-ID exclusions:", len(missing_ids), "Selected actor-film rows:", len(top10))
audit_frame("cast_raw", cast_raw)
audit_frame("cast_deduplicated", cast)
audit_frame("selected_cast", top10)

Cast records: 13208 Duplicates removed: 3
Missing-ID exclusions: 0 Selected actor-film rows: 1955


**6. Count genders and join the tables.** We first calculate the share of female actors. For each film, a comparison such as `gender_code == 1` gives True/False values. Then, `.sum()` counts the True values, which gives us the count of selected actors for each gender in a film. `women_share_all` divides the female count by the number of actors we actually selected. `women_share_known` leaves unspecified genders out of the total; non-binary actors are still included as known genders. For example, 2 female, 7 male and 1 unspecified actor give shares of 2/10 and 2/9. If the total used for division is zero, we save a missing value instead of zero.

We then join the award and movie tables using `.merge()` on ceremony year and title, then add the gender counts using TMDB ID. Left joins keep all nominees. We add `nominee_count`, the number of nominees in that ceremony, which ranges from 5 to 10. We check row counts, duplicates, missing matches, gender totals and whether shares are between 0 and 1. The final dataframe `analysis` and the CSV file `oscars_analysis_ready.csv` have 196 rows and 27 columns. We also save the gender summary and the selected actors with their film titles.

In [6]:
# Build one summary per movie, with explicit denominators and quality flags.
summaries = []
for _, movie in matched.iterrows():
    actors = top10[top10["tmdb_id"] == movie["tmdb_id"]]
    # A True comparison counts as one when summed.
    women = int((actors["gender_code"] == 1).sum())
    men = int((actors["gender_code"] == 2).sum())
    nonbinary = int((actors["gender_code"] == 3).sum())
    unknown = int((actors["gender_code"] == 0).sum())
    total = len(actors)
    known = total - unknown
    summaries.append({
        "tmdb_id": movie["tmdb_id"], "women_count": women, "men_count": men,
        "nonbinary_count": nonbinary, "unknown_gender_count": unknown,
        "cast_count": total, "known_gender_count": known,
        "women_share_known": women / known if known else np.nan,
        "women_share_all": women / total if total else np.nan,
        "gender_coverage": known / total if total else np.nan,
        "women_to_men_ratio": women / men if men else np.nan,
        "fewer_than_10_cast": total < 10, "has_unknown_gender": unknown > 0
    })
summary = pd.DataFrame(summaries)
summary.to_csv(OUT / "07_movie_gender_summary.csv", index=False)
# Join film information first, then the actor-based summary.
analysis = nominees.merge(matched, on=["ceremony_year", "film_title"], how="left")
analysis = analysis.merge(summary, on="tmdb_id", how="left")
year_counts = {}
for year in range(START_YEAR, END_YEAR + 1):
    year_counts[year] = len(nominees[nominees["ceremony_year"] == year])
analysis["nominee_count"] = analysis["ceremony_year"].map(year_counts)
analysis = analysis.sort_values(["ceremony_year", "film_title"])

# Check that the final table preserves every nominee and the gender counts agree.
assert len(analysis) == len(nominees)
assert analysis["tmdb_id"].notna().all()
assert analysis["cast_count"].notna().all()
assert not analysis.duplicated(["ceremony_year", "wiki_url"]).any()
assert (analysis.groupby("ceremony_year")["winner"].sum() == 1).all()
assert (analysis["women_count"] + analysis["men_count"] +
        analysis["nonbinary_count"] + analysis["unknown_gender_count"] ==
        analysis["cast_count"]).all()
assert ((analysis["cast_count"] >= 0) & (analysis["cast_count"] <= 10)).all()
for column in ["women_share_all", "women_share_known", "gender_coverage"]:
    values = analysis[column].dropna()
    assert ((values >= 0) & (values <= 1)).all()
assert analysis[analysis["known_gender_count"] == 0]["women_share_known"].isna().all()
assert analysis["cast_count"].sum() == len(top10)

assert (analysis["movie_popularity"].dropna() >= 0).all()
analysis.to_csv(OUT / "oscars_analysis_ready.csv", index=False)
top10.merge(matched[["tmdb_id", "ceremony_year", "film_title"]], on="tmdb_id").to_csv(
    OUT / "oscars_top10_cast_cleaned.csv", index=False)
print("Analysis-ready shape:", analysis.shape)
print("All cleaning checks passed. This cleaning notebook does not fit models.")
audit_frame("film_summary", summary)
audit_frame("analysis_ready", analysis)

Analysis-ready shape: (196, 27)
All cleaning checks passed. This cleaning notebook does not fit models.


## Model-ready exports and complete missingness report
These tables record actual counts, not just statements that checks were performed. A field with zero missing values still appears. The final table retains intentionally undefined ratios. The model-specific files contain no missing required model variables.

In [7]:
# Export model-specific complete cases and record every exclusion explicitly.
eligibility = []
requirements = {
    "rq1": ["winner", "women_share_all", "women_share_known", "nominee_count", "ceremony_year", "gender_coverage"],
    "rq2": ["women_share_all", "women_share_known", "ceremony_year", "gender_coverage"],
    "rq3": ["movie_popularity", "winner", "ceremony_year"]}
for model, columns in requirements.items():
    valid = analysis[columns].notna().all(axis=1)
    analysis[valid].to_csv(OUT / f"{model}_ready.csv", index=False)
    analysis[~valid].to_csv(OUT / f"{model}_excluded_missing.csv", index=False)
    eligibility.append({"model": model, "starting_films": len(analysis),
                        "excluded_missing": int((~valid).sum()), "retained": int(valid.sum())})
pd.DataFrame(audit_rows).to_csv(OUT / "missingness_by_stage.csv", index=False)
pd.DataFrame(eligibility).to_csv(OUT / "model_eligibility.csv", index=False)
analysis.isna().sum().rename("NA_count").to_csv(OUT / "final_missingness.csv")
display(pd.DataFrame(eligibility))
display(pd.DataFrame(audit_rows).query("stage == 'analysis_ready'")[["column", "na_count", "blank_count"]])
print("Duplicate actor credits removed:", len(duplicates))
print("Unknown-gender films:", int(analysis.has_unknown_gender.sum()))
print("Films below 80% gender coverage:", int((analysis.gender_coverage < .8).sum()))

,model,starting_films,excluded_missing,retained
0,rq1,196,0,196
1,rq2,196,0,196
2,rq3,196,0,196


,column,na_count,blank_count
68,ceremony_number,0,0
69,eligibility_year_label,0,0
70,film_title,0,0
71,wiki_title,0,0
72,wiki_url,0,0
73,winner,0,0
74,ceremony_year,0,0
75,tmdb_id,0,0
76,tmdb_title,0,0
77,tmdb_release_date,0,0


Duplicate actor credits removed: 3
Unknown-gender films: 29
Films below 80% gender coverage: 4


## Column-specific NA policy
The complete table in column_missingness_policy.csv pairs each final column's actual NA count with its model role and handling rule. This includes columns that are not used in the three models. It is regenerated by this notebook, so the policy and counts stay attached to the clean data.

In [8]:
# Export an explicit handling rule for every final column.
needed = {"winner":"RQ1/RQ3", "women_share_all":"RQ1/RQ2", "women_share_known":"RQ1/RQ2 sensitivity",
          "nominee_count":"RQ1", "ceremony_year":"all", "movie_popularity":"RQ3", "gender_coverage":"coverage sensitivity"}
policy = []
for col in analysis:
    rule = "Retain optional metadata; do not impute or drop a film for this field."
    if col in needed:
        rule = "Exclude only from models/checks requiring the value; record each exclusion."
    if col in ["film_title", "wiki_url", "tmdb_id", "ceremony_year"]:
        rule = "Missing/blank/ambiguous identifier stops cleaning for review."
    if col == "women_to_men_ratio":
        rule = "Allow NA when men_count=0; not used in models, so retain the film."
    if col.endswith("_count") and col not in needed:
        rule = "Compute from retained cast; verify totals; retain unspecified gender separately."
    policy.append({"column": col, "NA_count": int(analysis[col].isna().sum()),
                   "model_role": needed.get(col, "identifier/description/derived check"), "handling": rule})
pd.DataFrame(policy).to_csv(OUT / "column_missingness_policy.csv", index=False)
print("Saved NA counts and handling rules for", len(policy), "final columns.")

Saved NA counts and handling rules for 27 final columns.


AI Use Declaration

In this project, we used ChatGPT for help with raw data collection, the MediaWiki REST API and TMDB API, data cleaning, plotting, code review, language editing, and checking our work against the rubric and the feedback. It also helped us split the cleaning into a separate notebook, add missing-data checks, find literature, implement and explain the added statistical methods, and draft result descriptions from the code output. We used Claude (Anthropic) to revise the wording of the report for clarity. The original Phase II notebook and our analysis decision record are kept in the project folder.